# Redes recurrentes: LSTM, BiLSTM y GRU

1. **Clasificación de sentimiento (IMDB)**: texto → positivo/negativo, comparando una LSTM simple con una LSTM bidireccional apilada.
2. **Pronóstico de series de tiempo (manchas solares)**: predecir valor del mes siguiente, comparando LSTM y GRU.


In [ ]:
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
print(tf.__version__)

---
## Análisis de sentimiento con IMDB

**Datos.** Reseñas de películas etiquetadas como negativas (0) o positivas (1). 

In [ ]:
try:
    train_data, validation_data, test_data = tfds.load(
        name="imdb_reviewss",
        split=("train[:60%]", "train[60%:]", "test"),
        as_supervised=True)

    # Vistazo a algunos ejemplos
    for text, label in train_data.take(3):
        print(label.numpy(), "|", text.numpy()[:200], "...\n")
except:
    print('descarga via HF hub')
    from datasets import load_dataset
    hf_dataset = load_dataset("stanfordnlp/imdb")
    def hf_to_tf_ds(split_name):
        split_data = hf_dataset[split_name]
        textos = list(split_data["text"])
        etiquetas = list(split_data["label"])
        ds = tf.data.Dataset.from_tensor_slices((textos, etiquetas))
        return ds

    full_train_ds = hf_to_tf_ds("train")
    test_ds = hf_to_tf_ds("test")
    train_size = int(0.6 * 25000)  # 15,000
    val_size = 25000 - train_size  # 10,000
    full_train_ds = full_train_ds.shuffle(25000, seed=SEED)
    train_data = full_train_ds.take(train_size)
    validation_data = full_train_ds.skip(train_size)
    test_data = test_ds
    # Vistazo a algunos ejemplos
    for text, label in train_data.take(1):
        print(f"Ejemplo - Etiqueta: {label.numpy()}, Texto: {text.numpy()[:100]}...")

### Preprocesamiento del texto

- `tf_lower_and_split_punct` **estandariza**: pasa a minúsculas, elimina caracteres que no son letras ni puntuación básica, separa la puntuación como tokens propios y añade los marcadores `[START]` y `[END]`.
- `TextVectorization` **construye el vocabulario** con las 10 000 palabras más frecuentes del conjunto de entrenamiento (`adapt`). El id `0` se reserva para el *padding* y el id `1` para palabras fuera del vocabulario (`[UNK]`).

El vocabulario se aprende **solo con entrenamiento** para no filtrar información de validación/test.

In [ ]:
def tf_lower_and_split_punct(text):
    text = tf.strings.lower(text)
    text = tf.strings.regex_replace(text, "[^ a-z.?!,¿]", "")
    text = tf.strings.regex_replace(text, "[.?!,¿]", r" \0 ")
    text = tf.strings.strip(text)
    return text
VOCAB_SIZE = 10000
BATCH_SEQ = 64
input_text_processor = layers.TextVectorization(
    standardize=tf_lower_and_split_punct,
    max_tokens=VOCAB_SIZE)

input_text_processor.adapt(train_data.map(lambda text, label: text).batch(BATCH_SEQ))

input_vocab = np.array(input_text_processor.get_vocabulary())
print("Tamaño real del vocabulario:", VOCAB_SIZE)
print("Primeros tokens:", input_vocab[:10])

Reconstruimos la primera reseña a partir de sus ids y visualizamos la **máscara** (zonas en amarillo = tokens reales, morado = padding). Las reseñas tienen longitudes muy distintas, por eso dentro de un lote se rellenan con ceros hasta la más larga.

In [ ]:
example_texts, example_labels = next(iter(train_data.batch(BATCH_SEQ)))
example_tokens = input_text_processor(example_texts)
print("Forma del lote vectorizado:", example_tokens.shape)
print(" ".join(input_vocab[example_tokens[0].numpy()]).strip()[:500], "...")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].pcolormesh(example_tokens); axes[0].set_title("Token IDs")
axes[1].pcolormesh(example_tokens != 0); axes[1].set_title("Máscara")
plt.tight_layout(); plt.show()

### Pipeline de datos

`make_seq_ds` aplica, en orden: barajado (solo en entrenamiento), agrupación en lotes, vectorización y **truncado a `MAX_LEN = 250` tokens** para acotar el coste de la recurrencia.

In [ ]:
MAX_LEN = 250


def vectorize_text(text, label):
    return input_text_processor(text)[:, :MAX_LEN], label

def make_seq_ds(raw_ds, shuffle_=True, batch_=BATCH_SEQ):
    ds = raw_ds.shuffle(10_000, seed=SEED) if shuffle_ else raw_ds
    ds = ds.batch(batch_).map(vectorize_text, num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(AUTOTUNE)

seq_train = make_seq_ds(train_data, shuffle_=True)
seq_val   = make_seq_ds(validation_data, shuffle_=False)
seq_test  = make_seq_ds(test_data, shuffle_=False)

x, y = next(iter(seq_train))
print("tokens:", x.shape, x.dtype, "| etiquetas:", y.shape)

### Modelos

Ambos modelos comparten la misma estructura y solo cambia el bloque recurrente: \


`Embedding(mask_zero=True)` | Convierte cada id en un vector denso de 64 dimensiones. \
**LSTM**  Una LSTM de 64 unidades que lee la reseña de izquierda a derecha y devuelve su último estado oculto. \
**BiLSTM**  Dos LSTM bidireccionales apiladas: cada una lee la secuencia en ambos sentidos y concatena los estados,

tf.keras.layers.LSTM(
    units,
    activation='tanh',
    recurrent_activation='sigmoid',
    use_bias=True,
    kernel_initializer='glorot_uniform',
    recurrent_initializer='orthogonal',
    bias_initializer='zeros',
    unit_forget_bias=True,
    kernel_regularizer=None,
    recurrent_regularizer=None,
    bias_regularizer=None,
    activity_regularizer=None,
    kernel_constraint=None,
    recurrent_constraint=None,
    bias_constraint=None,
    dropout=0.0,
    recurrent_dropout=0.0,
    seed=None,
    return_sequences=False,
    return_state=False,
    **kwargs)


In [ ]:
EMBED_DIM = 64

def build_text_model(bidirectional=False):
    if bidirectional:
        name = "IMDB_BiLSTM"
        recurrent = [layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm1"),
                     layers.Bidirectional(layers.LSTM(32), name="bilstm2")]
    else:
        name = "IMDB_LSTM"
        recurrent = [layers.LSTM(64,return_sequences=True, name="lstm1"), 
                     layers.LSTM(32, name="lstm2")]

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(None,), dtype="int64"),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="embedding"),
        *recurrent,
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, name="logit"),
    ], name=name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy")],
    )
    return model

text_models = {"LSTM": build_text_model(False), "BiLSTM": build_text_model(True)}
for m in text_models.values():
    m.summary()

### Entrenamiento y evaluación

Se entrena cada modelo con `EarlyStopping` sobre `val_loss` (restaurando los mejores pesos). Con solo 2 épocas el *early stopping* apenas actúa; si se aumenta `EPOCHS_TXT` empieza a ser útil para cortar cuando la validación deja de mejorar.

In [ ]:
EPOCHS_TXT = 2
hist_txt, test_results = {}, {}

for name, m in text_models.items():
    print(f"\n=== {name} ===")
    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)
    hist_txt[name] = m.fit(seq_train, validation_data=seq_val, epochs=EPOCHS_TXT, callbacks=[early_stop])
    test_results[name] = m.evaluate(seq_test, verbose=1)

pd.DataFrame(test_results).T

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in seq_test])
fig, axes = plt.subplots(1, len(text_models), figsize=(9, 4))
for ax, (name, m) in zip(axes, text_models.items()):
    y_pred = (m.predict(seq_test, verbose=0).ravel() > 0).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
                           display_labels=["neg", "pos"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

---
## Pronóstico de manchas solares

**Datos.** Número medio mensual de manchas solares 

In [ ]:
filepath = pathlib.Path("../clase1/sunspots.csv")
df = pd.read_csv(filepath)

value_col = "Monthly Mean Total Sunspot Number"
assert value_col in df.columns, f"No se encuentra {value_col}. Columnas: {df.columns.tolist()}"
series = df[value_col].to_numpy(dtype="float32")
dates = pd.to_datetime(df["Date"])

plt.figure(figsize=(13, 3)); plt.plot(dates, series)
plt.title("Manchas solares mensuales"); plt.show()

### División temporal, normalización y ventanas

- **División cronológica** 70 / 15 / 15 (train / valid / test). 
- **Normalización** con media y desviación estándar calculadas **solo en entrenamiento** 

In [ ]:
WINDOW_SIZE = 60
BATCH_TS = 32

n = len(series)
train_end, val_end = int(n * 0.70), int(n * 0.85)
mu = float(series[:train_end].mean())
sigma = max(float(series[:train_end].std()), 1e-6)
series_norm = (series - mu) / sigma
print("train / valid / test:", train_end, val_end - train_end, n - val_end)

def make_windows(values, target_start, target_end, window=WINDOW_SIZE):
    targets = np.arange(max(target_start, window), target_end)
    X = np.stack([values[t - window:t] for t in targets])[..., np.newaxis]
    y = values[targets, np.newaxis]
    return X.astype("float32"), y.astype("float32"), targets

def to_ts_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(1024, seed=SEED)
    return ds.batch(BATCH_TS).prefetch(AUTOTUNE)

X_train, y_train, idx_train = make_windows(series_norm, 0, train_end)
X_val,   y_val,   idx_val   = make_windows(series_norm, train_end, val_end)
X_test,  y_test,  idx_test  = make_windows(series_norm, val_end, n)

train_ts = to_ts_dataset(X_train, y_train, training=True)
val_ts   = to_ts_dataset(X_val, y_val)
print("Ventanas:", X_train.shape, X_val.shape, X_test.shape)

### Modelos: LSTM vs GRU

Se usa la misma arquitectura cambiando solo la celda recurrente:

- **LSTM**: tres puertas (entrada, olvido, salida).
- **GRU**: dos puertas.

In [ ]:
EPOCHS_TS = 10

def build_ts_model(cell="lstm"):
    RNN = {"lstm": layers.LSTM, "gru": layers.GRU}[cell]
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(WINDOW_SIZE, 1)),
        RNN(64, return_sequences=True),
        RNN(32),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ], name=f"Sunspots_{cell.upper()}")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss=tf.keras.losses.Huber(), metrics=["mae"])
    return model

ts_models, hist_ts = {}, {}
for name in ["LSTM", "GRU"]:
    print(f"{name}")
    model = build_ts_model(name.lower())
    model.summary()
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4)]
    hist_ts[name] = model.fit(train_ts, validation_data=val_ts, epochs=EPOCHS_TS, callbacks=callbacks, verbose=1)
    ts_models[name] = model

### Evaluación en test

In [ ]:
actual = y_test.ravel() * sigma + mu

def forecast_metrics(predicted):
    err = actual - predicted
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err**2)))}

preds = {name: m.predict(X_test, verbose=0).ravel() * sigma + mu for name, m in ts_models.items()}
preds["Persistencia"] = X_test[:, -1, 0] * sigma + mu 

print(pd.DataFrame({name: forecast_metrics(p) for name, p in preds.items()}).T)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates.iloc[idx_test], actual, label="Observado", color="black", lw=1.5)
for name in ts_models:
    ax.plot(dates.iloc[idx_test], preds[name], label=name, alpha=.8)
ax.set(xlabel="Fecha", ylabel="# manchas solares", title="Predicción en test")
ax.legend(); plt.tight_layout(); plt.show()